## Training a Diabetes Prediction Model - Logistic Regression

### Setup the Azure ML Client

In [ ]:
from azure.ai.ml.entities import AzureBlobDatastore
from azure.ai.ml.entities import AccountKeyConfiguration
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

ml_client = MLClient.from_config(credential=DefaultAzureCredential())

### Connecting to Azure ML Workspace using the Azure ML SDK v2

In [ ]:
import azureml.core
from azureml.core import Workspace

# Load the workspace from the saved config file
ws = Workspace.from_config()
print('Ready to use Azure ML {} to work with {}'.format(azureml.core.VERSION, ws.name))

### Loading the Diabetes Dataset from the Data Asset

In [ ]:
from azureml.core import Workspace, Dataset

dataset = Dataset.get_by_name(ws, name='diabetes dataset')
diabetes_df = dataset.to_pandas_dataframe()

In [ ]:
display(diabetes_df)

### Normalizing and Scaling the Data

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

numericFeatures = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction"
]

# Extract the feature matrix (similar to VectorAssembler)
numericFeaturesVector = diabetes_df[numericFeatures]

# Apply MinMax Scaling
scaler = MinMaxScaler()

normalizedFeatures = scaler.fit_transform(numericFeaturesVector)

# Compare original vs scaled
compareNumerics = pd.DataFrame({
    "numericFeatures": numericFeaturesVector.values.tolist(),
    "normalizedFeatures": normalizedFeatures.tolist()
})

display(compareNumerics)

### Creating the Scaled Training and Test Sets

In [ ]:
# Create a scaled dataframe
scaledData = pd.DataFrame(
    normalizedFeatures,
    columns=numericFeatures
)

# Add the label column
scaledData["Outcome"] = diabetes_df["Outcome"]

display(scaledData)

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(scaledData, test_size=0.3, random_state=42)

print("Training Rows:", len(train), "Testing Rows:", len(test))

In [ ]:
X_train = train[numericFeatures]
Y_train = train['Outcome']

display(X_train)
display(Y_train)

### Training the Model  

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=10,
    C=1/0.3
)

model.fit(X_train, Y_train)

print("Model trained!")

### Getting Predictions from the Model

In [ ]:
# Scale test data (IMPORTANT: use transform, not fit_transform)
X_test = scaledData[numericFeatures]
Y_test = scaledData["Outcome"]
# Make predictions
predictions = model.predict(X_test)

# Get probabilities
probabilities = model.predict_proba(X_test)

# Create results dataframe
predicted = pd.DataFrame({
    "features": X_test.values.tolist(),
    "probability": probabilities.tolist(),
    "prediction": predictions,
    "trueLabel": Y_test.values
})

display(predicted)

### Evaluating the Model Performance

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(Y_test, predictions))